# Feature engineering Avanzato
## Fire count cell
numero di incendi che cade in una specifica cella geografica

In [ ]:
import pandas as pd

df = pd.read_csv('dataset_pulito.csv')

In [ ]:
df.shape

In [ ]:
df.head()

### Trasformazione coordinate in cella

In [ ]:
import numpy as np

grid_size = 0.1

df["lat_cell"] = (
    np.floor(df["latitude"] / grid_size) * grid_size
).round(4)

df["lon_cell"] = (
    np.floor(df["longitude"] / grid_size) * grid_size
).round(4)

In [ ]:
fire_counts = (
    df.groupby(["lat_cell", "lon_cell"])
      .agg(
          fire_count_cell=("latitude", "count"),
          frp_sum=("frp", "sum"),
          frp_mean=("frp", "mean")
      )
      .reset_index()
)

print(fire_counts)

In [ ]:
df = df.merge(
    fire_counts,
    on=["lat_cell", "lon_cell"],
    how="left"
)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.to_csv("dataset_pulito.csv", index=False)

In [ ]:
pd.read_csv("dataset_pulito.csv").shape

In [ ]:
df.head()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

grid_size = 0.5

fire_counts["geometry"] = fire_counts.apply(
    lambda row: box(
        row["lon_cell"],
        row["lat_cell"],
        row["lon_cell"] + grid_size,
        row["lat_cell"] + grid_size
    ),
    axis=1
)

gdf_grid = gpd.GeoDataFrame(
    fire_counts,
    geometry="geometry",
    crs="EPSG:4326"
)

fig, ax = plt.subplots(figsize=(12, 8))

gdf_grid.plot(
    column="fire_count_cell",
    ax=ax,
    legend=True,
    cmap="hot",
    edgecolor="none"
)

ax.set_title("Fire Count per Grid Cell", fontsize=16)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()

# FRP 

l'FRP e' il Fire Radaitive Power,ovvero, la quantita' di radioattivita' emessa dal fuoco. 

Precedentemente e' stato calcolato l'frp medio in una specifica area, ora calcolo quello massimo in un area e quello anomalo

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('dataset_pulito.csv') 

## FRP max

frp_max = max(frp)

In [ ]:
features = (
    df.groupby(["lat_cell", "lon_cell"])
      .agg(
          frp_mean=("frp", "mean"),
          frp_max=("frp", "max"),
          frp_std=("frp", "std"),
          frp_sum=("frp", "sum"),
          fire_count=("frp", "count"),
      )
      .reset_index()
)

features["frp_std"] = features["frp_std"].replace(0, np.nan)

## Anomaly FRP

$anomaly = \frac{frp - frp_mean}{frp_std}$

In [ ]:
features["frp_anomaly_score"] = (
    (features["frp_max"] - features["frp_mean"]) /
    features["frp_std"]
)

features["frp_anomaly_score"] = (
    features["frp_anomaly_score"]
      .fillna(0)
)

### Salvataggio dataset

In [ ]:
df = df.merge(
    features,
    on=["lat_cell", "lon_cell"],
    how="left"
)

# salva mantenendo tutte le colonne originali
df.to_csv("dataset_pulito2.csv", index=False)

print(df.head())

## Feature temporali derivanti da acq_date

In [ ]:
import pandas as pd
import numpy as np
INPUT_FILE = "dataset_pulito2.csv"


df = pd.read_csv(INPUT_FILE)


df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")

df["month"] = df["acq_date"].dt.month.astype("Int8")

df["day_of_year"] = df["acq_date"].dt.dayofyear.astype("Int16")

df["week"] = df["acq_date"].dt.isocalendar().week.astype("Int16")

df["sin_doy"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["cos_doy"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

df.drop(columns=["acq_date"], inplace=True)


df.to_csv("dataset_pulito3.csv", index=False)

print("Feature engineering completato.")

## INIZIO BLOCCO AGGIUNTO: lag + target post-FRP

Da questo punto il notebook continua subito dopo il calcolo FRP gia fatto sopra.

- non richiama file `.py` esterni
- non ricalcola FRP
- calcola prima i lag storici e poi i target futuri


In [ ]:
# =========================
# INIZIO BLOCCO LAG POST-FRP
# Parte dal dataset post-FRP gia presente nel notebook.
# Non ricalcola FRP: costruisce solo base giornaliera e lag storici.
# =========================

import pandas as pd
import numpy as np
from pathlib import Path

CELL_COLUMNS = ["lat_cell", "lon_cell"]
LAG_COLUMNS = [
    "fire_count_last_1d",
    "fire_count_last_3d",
    "fire_count_last_7d",
    "frp_mean_last_7d",
    "days_since_last_fire",
]
FINAL_OUTPUT = Path("dataset_pulito3_lag_target_final.csv")

# Qui parte la continuazione post-FRP: base giornaliera + lag storici.
source_df = df.copy() if "df" in globals() else pd.read_csv("dataset_pulito3.csv")

date = pd.Series(pd.NaT, index=source_df.index, dtype="datetime64[ns]")
for column in ["chunk_start", "chunk_end", "acq_date", "date"]:
    if column in source_df.columns:
        date = date.fillna(pd.to_datetime(source_df[column], errors="coerce"))

source_df = source_df.assign(
    date=date.dt.normalize(),
    frp=pd.to_numeric(source_df["frp"], errors="coerce"),
)

daily_df = (
    source_df.drop_duplicates()
    .dropna(subset=CELL_COLUMNS + ["date"])
    .groupby(CELL_COLUMNS + ["date"], as_index=False)
    .agg(
        fire_count=("frp", "size"),
        daily_frp_mean=("frp", "mean"),
        source_rows=("frp", "size"),
    )
    .sort_values(CELL_COLUMNS + ["date"], ignore_index=True)
)

full_grid = (
    daily_df[CELL_COLUMNS]
    .drop_duplicates()
    .merge(
        pd.DataFrame({"date": pd.date_range(daily_df["date"].min(), daily_df["date"].max(), freq="D")}),
        how="cross",
    )
)

lag_df = (
    full_grid.merge(daily_df, on=CELL_COLUMNS + ["date"], how="left", validate="one_to_one")
    .sort_values(CELL_COLUMNS + ["date"], ignore_index=True)
)

lag_df["fire_count"] = lag_df["fire_count"].fillna(0).astype("int32")
lag_df["source_rows"] = lag_df["source_rows"].fillna(0).astype("int32")
lag_df["month"] = lag_df["date"].dt.month.astype("Int8")
lag_df["day_of_year"] = lag_df["date"].dt.dayofyear.astype("Int16")
lag_df["week"] = lag_df["date"].dt.isocalendar().week.astype("Int16")
lag_df["sin_doy"] = np.sin(2 * np.pi * lag_df["day_of_year"].astype(float) / 365.0)
lag_df["cos_doy"] = np.cos(2 * np.pi * lag_df["day_of_year"].astype(float) / 365.0)

g = lag_df.groupby(CELL_COLUMNS, sort=False)
history_len = g.cumcount()
past_fire = pd.concat([g["fire_count"].shift(i).rename(i) for i in range(1, 8)], axis=1).astype("Float64")
past_frp = pd.concat([g["daily_frp_mean"].shift(i).rename(i) for i in range(1, 8)], axis=1).astype("Float64")

last_fire_date = lag_df["date"].where(lag_df["fire_count"] > 0)
cell_keys = [lag_df["lat_cell"], lag_df["lon_cell"]]
prev_fire_date = last_fire_date.groupby(cell_keys, sort=False).ffill().groupby(cell_keys, sort=False).shift(1)

lag_df["fire_count_last_1d"] = past_fire[1].round().astype("Int64")
lag_df["fire_count_last_3d"] = past_fire[[1, 2, 3]].sum(axis=1, min_count=3).round().astype("Int64")
lag_df["fire_count_last_7d"] = past_fire.sum(axis=1, min_count=7).round().astype("Int64")
lag_df["frp_mean_last_7d"] = past_frp.mean(axis=1).astype("Float64")
lag_df["days_since_last_fire"] = (lag_df["date"] - prev_fire_date).dt.days.astype("Int64")

lag_df.loc[history_len < 1, "fire_count_last_1d"] = pd.NA
lag_df.loc[history_len < 3, "fire_count_last_3d"] = pd.NA
lag_df.loc[history_len < 7, ["fire_count_last_7d", "frp_mean_last_7d"]] = pd.NA

print("Missing values in lag columns:")
print(lag_df[LAG_COLUMNS].isna().sum())

lag_df.head()


In [ ]:
# ============================
# CONTINUAZIONE BLOCCO TARGET
# Da qui aggiungiamo i target futuri partendo dai lag calcolati sopra.
# ============================

TARGET_COLUMNS = [
    "fire_next_1d",
    "fire_next_3d",
    "fire_next_7d",
    "fire_count_next_7d",
]


def to_binary(series):
    out = pd.Series(pd.NA, index=series.index, dtype="Int8")
    mask = series.notna()
    out.loc[mask] = series.loc[mask].gt(0).astype("int8")
    return out


g = lag_df.groupby(CELL_COLUMNS, sort=False)
future_fire = pd.concat([g["fire_count"].shift(-i).rename(i) for i in range(1, 8)], axis=1).astype("Float64")

next_1d = future_fire[1]
next_3d = future_fire[[1, 2, 3]].sum(axis=1, min_count=3)
next_7d = future_fire.sum(axis=1, min_count=7)

# Qui si chiude il blocco aggiunto: target futuri + salvataggio CSV finale.
final_temporal_df = lag_df.copy()
final_temporal_df["fire_next_1d"] = to_binary(next_1d)
final_temporal_df["fire_next_3d"] = to_binary(next_3d)
final_temporal_df["fire_next_7d"] = to_binary(next_7d)
final_temporal_df["fire_count_next_7d"] = next_7d.round().astype("Int64")

final_temporal_df = final_temporal_df[
    [
        "date", "lat_cell", "lon_cell", "fire_count", "daily_frp_mean", "source_rows",
        "month", "day_of_year", "week", "sin_doy", "cos_doy",
        "fire_count_last_1d", "fire_count_last_3d", "fire_count_last_7d",
        "frp_mean_last_7d", "days_since_last_fire",
        "fire_next_1d", "fire_next_3d", "fire_next_7d", "fire_count_next_7d",
    ]
].copy()

final_temporal_df.to_csv(FINAL_OUTPUT, index=False)

print(f"Final dataset shape: {final_temporal_df.shape}")
print(f"Saved file: {FINAL_OUTPUT.resolve()}")
print("Missing values in target columns:")
print(final_temporal_df[TARGET_COLUMNS].isna().sum())

final_temporal_df.head()


## FINE BLOCCO AGGIUNTO: lag + target post-FRP

Da qui puoi riprendere con il resto del notebook. `final_temporal_df` e il CSV finale sono gia pronti.


In [ ]:
df = pd.read_csv("dataset_pulito3.csv")
df.head()

# Fire density
Stima continua della densita' degli incendi nello spazio

density(x) = somma delle influenze di tutti i punti vicini. Ogni punto influisce con una gaussiana

In [ ]:
coords = df[["latitude", "longitude"]].values

In [ ]:
from sklearn.neighbors import KernelDensity

kde = KernelDensity(
    bandwidth=0.2,  
    kernel="gaussian"
)

kde.fit(coords)

In [22]:
import numpy as np

density = np.exp(kde.score_samples(coords))

df["fire_density"] = density

KeyboardInterrupt: 

In [ ]:
df.head()